# SupplyMind — Google Colab Unified Model Training
**Models included: LightGBM (Supplier Risk), LSTM Autoencoder (Anomalies), TFT (Demand Forecasting)**

### Instructions:
1. Open this notebook in Google Colab.
2. In Colab, change runtime type to **T4 GPU** (`Runtime -> Change runtime type -> T4 GPU`).
3. Compress your local `data/processed` folder into a zip file (`processed.zip`) and upload it to Colab's file explorer, or mount Google Drive.
4. Run all cells. Once completed, download the saved checkpoints (`best.joblib`, `best_lstm.pt`, and `tft_best.ckpt`) back to your local model directories.

In [ ]:
# 1. Setup & Install Dependencies in Colab
!pip install lightgbm optuna imbalanced-learn pytorch-forecasting mlflow joblib loguru pyarrow

# Unzip data if you uploaded processed.zip
import os
if os.path.exists('processed.zip'):
    !unzip -q processed.zip -d data/
    print("Data unpacked successfully!")
else:
    print("Please upload processed.zip containing your supplier_train/val/test and demand_features parquets.")

---  
## MODEL 1: LightGBM Supplier Risk Model

In [ ]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
import pandas as pd
import numpy as np
import optuna
import lightgbm as lgb
import joblib
from imblearn.combine import SMOTETomek
from sklearn.calibration import CalibratedClassifierCV
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold

CATEGORICAL_FEATURES = ["country", "contract_tier"]
NUMERIC_FEATURES = [
    "tenure_years", "prev_otd", "prev_lead_time", "prev_po_accept",
    "otd_mean_4w", "otd_mean_12w", "lead_time_mean_4w", "lead_time_std_4w",
    "lead_time_mean_12w", "po_accept_mean_4w", "lead_time_slope_6w"
]
TARGET = "disruption_flag"

encoders = {}

def load_and_prepare_train(path):
    df = pd.read_parquet(path)
    for col in CATEGORICAL_FEATURES:
        le = LabelEncoder()
        df[col] = le.fit_transform(df[col].astype(str))
        encoders[col] = le
    return df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy(), df[TARGET].astype(int)

def load_and_prepare_val(path):
    df = pd.read_parquet(path)
    for col in CATEGORICAL_FEATURES:
        le = encoders[col]
        known_classes = set(le.classes_)
        # handle unseen labels safely by mapping them to the first known class
        df[col] = df[col].apply(lambda x: x if x in known_classes else le.classes_[0])
        df[col] = le.transform(df[col].astype(str))
    return df[NUMERIC_FEATURES + CATEGORICAL_FEATURES].copy(), df[TARGET].astype(int)

def _objective(trial, X_train, y_train, cat_indices):
    params = {
        "objective": "binary",
        "metric": "average_precision",
        "verbosity": -1,
        "num_leaves": trial.suggest_int("num_leaves", 31, 128),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
        "min_child_samples": trial.suggest_int("min_child_samples", 10, 50),
        "feature_fraction": trial.suggest_float("feature_fraction", 0.6, 1.0),
        "bagging_fraction": trial.suggest_float("bagging_fraction", 0.6, 1.0),
        "bagging_freq": trial.suggest_int("bagging_freq", 1, 5),
        "n_estimators": trial.suggest_int("n_estimators", 100, 500),
    }
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []
    for fold_train_idx, fold_val_idx in skf.split(X_train, y_train):
        X_f_tr, X_f_val = X_train[fold_train_idx], X_train[fold_val_idx]
        y_f_tr, y_f_val = y_train[fold_train_idx], y_train[fold_val_idx]
        model = lgb.LGBMClassifier(**params)
        model.fit(
            X_f_tr, y_f_tr,
            eval_set=[(X_f_val, y_f_val)],
            categorical_feature=cat_indices,
            callbacks=[lgb.early_stopping(30, verbose=False)]
        )
        proba = model.predict_proba(X_f_val)[:, 1]
        scores.append(average_precision_score(y_f_val, proba))
    return float(np.mean(scores))

print("Loading LightGBM data...")
X_train_raw, y_train_raw = load_and_prepare_train("data/processed/supplier_train.parquet")
X_val, y_val = load_and_prepare_val("data/processed/supplier_val.parquet")
cat_indices = [X_train_raw.columns.get_loc(c) for c in CATEGORICAL_FEATURES]

print("Applying SMOTE-Tomek...")
smt = SMOTETomek(random_state=42)
X_train, y_train = smt.fit_resample(X_train_raw.values, y_train_raw.values)
# Round categorical columns generated by SMOTE to integers
for idx in cat_indices:
    X_train[:, idx] = np.round(X_train[:, idx]).astype(int)

print("Running Optuna HPO...")
study = optuna.create_study(direction="maximize")
study.optimize(lambda trial: _objective(trial, X_train, y_train, cat_indices), n_trials=15)

final_model = lgb.LGBMClassifier(**study.best_params, objective="binary", verbosity=-1)
final_model.fit(X_train, y_train, categorical_feature=cat_indices)

calibrated = CalibratedClassifierCV(final_model, method="isotonic", cv="prefit")
calibrated.fit(X_val.values, y_val.values)

proba_val = calibrated.predict_proba(X_val.values)[:, 1]
print(f"LightGBM Trained! Val PR-AUC: {average_precision_score(y_val, proba_val):.4f} | ROC-AUC: {roc_auc_score(y_val, proba_val):.4f}")
joblib.dump({"model": calibrated, "feature_names": X_train_raw.columns.tolist()}, "best.joblib")

---  
## MODEL 2: LSTM Autoencoder Anomaly Detection Model

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
import joblib
import json
import numpy as np
import pandas as pd
from scipy.stats import genpareto

# Features matching the updated local train.py
SEQ_LENGTH = 12
FEATURES = [
    "prev_lead_time", "lead_time_cv", "prev_otd", "defect_rate",
    "financial_stress_score", "capacity_utilization",
    "regional_delay_factor", "port_congestion_index", "weather_alerts",
    "interest_rate", "inflation_index", "raw_material_cost",
    "lead_time_volatility_4w", "lead_time_volatility_12w"
]

class Attention(nn.Module):
    def __init__(self, encoder_dim: int, decoder_dim: int):
        super().__init__()
        self.attn = nn.Linear(encoder_dim + decoder_dim, decoder_dim)
        self.v = nn.Linear(decoder_dim, 1, bias=False)

    def forward(self, decoder_hidden, encoder_outputs):
        seq_len = encoder_outputs.size(1)
        decoder_hidden_expanded = decoder_hidden.unsqueeze(1).repeat(1, seq_len, 1)
        energy = torch.tanh(self.attn(torch.cat((decoder_hidden_expanded, encoder_outputs), dim=2)))
        attention_scores = self.v(energy).squeeze(2)
        return torch.softmax(attention_scores, dim=1)

class LSTMAutoencoder(nn.Module):
    def __init__(self, n_features: int, hidden_dim: int = 32, latent_dim: int = 16, num_layers: int = 2, bidirectional: bool = True):
        super().__init__()
        self.n_features = n_features
        self.hidden_dim = hidden_dim
        self.latent_dim = latent_dim
        self.num_layers = num_layers
        self.bidirectional = bidirectional
        
        self.encoder = nn.LSTM(
            input_size=n_features,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=0.2 if num_layers > 1 else 0.0,
            bidirectional=bidirectional,
            batch_first=True
        )
        
        encoder_out_dim = hidden_dim * 2 if bidirectional else hidden_dim
        self.bottleneck = nn.Linear(encoder_out_dim, latent_dim)
        
        self.decoder_hidden_init = nn.Linear(latent_dim, hidden_dim)
        self.decoder_cell_init = nn.Linear(latent_dim, hidden_dim)
        
        self.decoder = nn.LSTM(
            input_size=n_features + encoder_out_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            dropout=0.2 if num_layers > 1 else 0.0,
            batch_first=True
        )
        
        self.attention = Attention(encoder_dim=encoder_out_dim, decoder_dim=hidden_dim)
        self.reconstruct = nn.Linear(hidden_dim, n_features)

    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        enc_out, (hn, cn) = self.encoder(x)
        
        if self.bidirectional:
            last_hn = torch.cat((hn[-2], hn[-1]), dim=1)
        else:
            last_hn = hn[-1]
            
        latent = self.bottleneck(last_hn)
        
        dec_h = self.decoder_hidden_init(latent).unsqueeze(0).repeat(self.num_layers, 1, 1)
        dec_c = self.decoder_cell_init(latent).unsqueeze(0).repeat(self.num_layers, 1, 1)
        
        outputs = []
        dec_input = torch.zeros(batch_size, 1, self.n_features, device=x.device)
        
        for t in range(seq_len):
            attn_weights = self.attention(dec_h[-1], enc_out)
            context = torch.bmm(attn_weights.unsqueeze(1), enc_out)
            decoder_input_combined = torch.cat((dec_input, context), dim=2)
            dec_out, (dec_h, dec_c) = self.decoder(decoder_input_combined, (dec_h, dec_c))
            reconstruction = self.reconstruct(dec_out)
            outputs.append(reconstruction)
            dec_input = reconstruction
            
        reconstructed_seq = torch.cat(outputs, dim=1)
        return reconstructed_seq

class EarlyStopping:
    def __init__(self, patience: int = 5, min_delta: float = 0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_loss = None
        self.early_stop = False

    def __call__(self, val_loss):
        if self.best_loss is None:
            self.best_loss = val_loss
        elif val_loss > self.best_loss - self.min_delta:
            self.counter += 1
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_loss = val_loss
            self.counter = 0

def fit_pot_threshold(errors: np.ndarray, quantile: float = 0.90, extreme_quantile: float = 0.95) -> float:
    initial_threshold = np.quantile(errors, quantile)
    excesses = errors[errors > initial_threshold] - initial_threshold
    if len(excesses) < 10:
        return float(np.quantile(errors, extreme_quantile))
    
    try:
        c, _, scale = genpareto.fit(excesses, floc=0)
        n = len(errors)
        n_t = len(excesses)
        prob = 1 - extreme_quantile
        if abs(c) < 1e-6:
            extreme_threshold = initial_threshold + scale * np.log(n_t / (n * prob))
        else:
            extreme_threshold = initial_threshold + (scale / c) * (((n * prob) / n_t) ** (-c) - 1)
        
        if not np.isfinite(extreme_threshold) or extreme_threshold < initial_threshold:
            return float(np.quantile(errors, extreme_quantile))
        return float(extreme_threshold)
    except Exception:
        return float(np.quantile(errors, extreme_quantile))

print("Preparing LSTM Autoencoder sequences...")
full_supplier_train = pd.read_parquet("data/processed/supplier_train.parquet")
healthy_train = full_supplier_train[full_supplier_train["disruption_flag"] == 0].copy()

scaler = StandardScaler()
healthy_train[FEATURES] = scaler.fit_transform(healthy_train[FEATURES])

X_train_seq = []
healthy_train = healthy_train.sort_values(["supplier_id", "week_num"])
for supplier, group in healthy_train.groupby("supplier_id"):
    vals = group[FEATURES].values
    if len(vals) >= SEQ_LENGTH:
        for i in range(len(vals) - SEQ_LENGTH + 1):
            X_train_seq.append(vals[i : i+SEQ_LENGTH])

X_train_seq = np.array(X_train_seq, dtype=np.float32)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
dataset = TensorDataset(torch.tensor(X_train_seq))
loader = DataLoader(dataset, batch_size=256, shuffle=True)

model = LSTMAutoencoder(n_features=len(FEATURES)).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3, verbose=True)
early_stopping = EarlyStopping(patience=5)
criterion = nn.MSELoss()

print("Training LSTM Autoencoder...")
best_loss = float("inf")
best_model_state = None

for epoch in range(30):
    model.train()
    total_loss = 0
    for batch in loader:
        inputs = batch[0].to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, inputs)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * inputs.size(0)
    epoch_loss = total_loss / len(loader.dataset)
    print(f"Epoch {epoch+1} Loss: {epoch_loss:.5f}")
    
    scheduler.step(epoch_loss)
    early_stopping(epoch_loss)
    
    if epoch_loss < best_loss:
        best_loss = epoch_loss
        best_model_state = model.state_dict()
        
    if early_stopping.early_stop:
        print("Early stopping triggered.")
        break

model.load_state_dict(best_model_state)
model.eval()

# Calculate POT Threshold
train_errors = []
with torch.no_grad():
    for batch in loader:
        inputs = batch[0].to(device)
        outputs = model(inputs)
        err = nn.MSELoss(reduction='none')(outputs, inputs).mean(dim=(1, 2))
        train_errors.extend(err.cpu().numpy())

train_errors = np.array(train_errors)
anomaly_threshold = fit_pot_threshold(train_errors, quantile=0.90, extreme_quantile=0.95)
print(f"Fitted POT threshold: {anomaly_threshold:.6f}")

# Save artifacts
torch.save(best_model_state, "best_lstm.pt")
joblib.dump(scaler, "scaler.joblib")
with open("threshold.json", "w") as f:
    json.dump({"anomaly_threshold": anomaly_threshold}, f)
print("LSTM Autoencoder Trained and Saved!")

---  
## MODEL 3: Temporal Fusion Transformer (TFT) Demand Forecasting Model

In [ ]:
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer
from pytorch_forecasting.metrics import QuantileLoss
import pytorch_lightning as pl
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint

print("Loading TFT dataset...")
df_demand = pd.read_parquet("data/processed/demand_features.parquet")

# Ensure time_idx is integer sequential days
df_demand["time_idx"] = (df_demand["date"] - df_demand["date"].min()).dt.days
df_demand = df_demand.sort_values(by=["id", "time_idx"]).reset_index(drop=True)

# Convert categories to strings
df_demand["id"] = df_demand["id"].astype(str)
df_demand["item_id"] = df_demand["item_id"].astype(str)
df_demand["dept_id"] = df_demand["dept_id"].astype(str)
df_demand["cat_id"] = df_demand["cat_id"].astype(str)
df_demand["event_name_1"] = df_demand["event_name_1"].astype(str)

max_prediction_length = 14
max_encoder_length = 28
training_cutoff = df_demand["time_idx"].max() - max_prediction_length

training_dataset = TimeSeriesDataSet(
    df_demand[df_demand["time_idx"] <= training_cutoff],
    time_idx="time_idx",
    target="sales",
    group_ids=["id"],
    min_encoder_length=max_encoder_length // 2,
    max_encoder_length=max_encoder_length,
    min_prediction_length=1,
    max_prediction_length=max_prediction_length,
    static_categoricals=["item_id", "dept_id", "cat_id"],
    time_varying_known_categoricals=["event_name_1"],
    time_varying_known_reals=["time_idx", "snap_CA"],
    time_varying_unknown_reals=[
        "sales", "lag_1", "lag_7", "lag_14", "lag_28",
        "rolling_mean_7", "rolling_mean_28", "rolling_std_7"
    ],
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

validation_dataset = TimeSeriesDataSet.from_dataset(training_dataset, df_demand, predict=True, stop_randomization=True)

train_dataloader = training_dataset.to_dataloader(batch_size=64, train=True, num_workers=2)
val_dataloader = validation_dataset.to_dataloader(batch_size=128, train=False, num_workers=2)

pl.seed_everything(42)

early_stop_callback = EarlyStopping(monitor="val_loss", min_delta=1e-4, patience=3, verbose=False, mode="min")
checkpoint_callback = ModelCheckpoint(monitor="val_loss", filename="tft-checkpoint", save_top_k=1, mode="min")

trainer = pl.Trainer(
    max_epochs=5,
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    enable_model_summary=True,
    callbacks=[early_stop_callback, checkpoint_callback],
)

tft = TemporalFusionTransformer.from_dataset(
    training_dataset,
    learning_rate=0.03,
    hidden_size=16,
    attention_head_size=4,
    dropout=0.1,
    loss=QuantileLoss([0.05, 0.5, 0.95]),
    reduce_on_plateau_patience=2,
)

print("Training TFT model (using GPU if available)...")
trainer.fit(tft, train_dataloaders=train_dataloader, val_dataloaders=val_dataloader)

best_model_path = trainer.checkpoint_callback.best_model_path
print(f"TFT Trained! Best model saved to: {best_model_path}")
# Copy model to current folder for easy download
import shutil
shutil.copy(best_model_path, "tft_best.ckpt")
print("tft_best.ckpt copied to workspace.")